In [1]:
import numpy as np 
import pandas as pd 

### MLP Class 

In [18]:
class MLP:
    def __init__(self,architecture,X,y,epochs,lr,use_case):
        self.architecture = architecture
        self.X = X 
        self.y = y 
        self.lr = lr 
        self.use_case = use_case
        self.outputs = {}
        self.raw_outputs = []
        self.gradients = {}
        self.epochs = epochs

        self.W_b = []
        for i in range(1,len(architecture)):
            weights = np.random.rand(architecture[i],architecture[i-1])
            bias = np.zeros(architecture[i])
            self.W_b.append((weights,bias))

    def relu(self,z):
        return np.maximum(0,z)

    def sigmoid(self,z):
        return 1/(1+np.exp(-z))

    def softmax(self,y_pred):
        exp = np.exp(y_pred - np.max(y_pred,axis=1,keepdims=True))
        return exp/np.sum(exp,axis=1,keepdims=True)

    def derivative_sigmoid(self,z):
        return self.sigmoid(z)*(1-self.sigmoid(z))

    def derivative_relu(self,z):
        return (z>0).astype(int)

    def forward(self):
        A = self.X 
        self.raw_outputs = []
        self.outputs = {}
        self.raw_outputs.append(A)

        for i in range(len(self.W_b)):
            W,b = self.W_b[i]
            Z = A @ W.T + b

            self.raw_outputs.append(Z)
            if i == len(self.W_b)-1:
                if self.use_case == "Classification":
                    A = self.softmax(Z)
                else:
                    A = Z 
            else:
                A = self.relu(Z)
            self.outputs[i]=A 
        return A 

    def backward(self,y_pred):
        if self.use_case == "Regression":
            DL_DZ = (2/self.y.shape[0]) * (y_pred - self.y) 
        if self.use_case == "Classification":
            y_one_hot = np.zeros_like(y_pred)

            y_one_hot[np.arange(len(self.y)), self.y] = 1

            # Softmax + Cross Entropy gradient
            DL_DZ = (y_pred - y_one_hot) / self.y.shape[0]
        for i in range(len(self.W_b)-1,-1,-1):
            if i==0:
                A_prev = self.X
            else:
                A_prev = self.outputs[i-1]
            DL_DW = DL_DZ.T @ A_prev
            DL_DB = np.sum(DL_DZ,axis=0)
            self.gradients[i]={
                "weights": DL_DW,
                "bias"   : DL_DB
            }
            if i>0:
                DL_DA = DL_DZ @ self.W_b[i][0]
                DL_DZ = DL_DA  * self.derivative_relu(self.raw_outputs[i])

    def update(self):
        for i in range(len(self.W_b)):
            W,b = self.W_b[i]
            W = W - self.lr * self.gradients[i]['weights']
            b = b - self.lr * self.gradients[i]['bias']
            self.W_b[i] = (W,b)

    def loss(self,y_pred):
        if self.use_case == "Regression":
            return np.mean((y_pred-self.y)**2)
        if self.use_case == "Classification":
            return -np.mean(np.log(y_pred[np.arange(len(self.y)),self.y]+1e-15))

    def fit(self):
        for _ in range(self.epochs):
            y_pred = self.forward()
            loss = self.loss(y_pred)
            self.backward(y_pred)
            self.update()

### Regression Using MLP

In [19]:
import numpy as np

np.random.seed(42)

# 100 samples, 2 features
X = np.random.randn(100, 2)

# True relationship
y = 3 * X[:, 0] + 2 * X[:, 1] + 5

# Add small noise
y += np.random.randn(100) * 0.2

# Make y shape (100, 1)
y = y.reshape(-1, 1)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (100, 2)
y shape: (100, 1)


In [20]:
model = MLP(
    architecture=[2, 8, 4, 1],
    X=X,
    y=y,
    epochs=5000,
    lr=0.01,
    use_case="Regression"
)
model.fit()

y_pred = model.forward()

print("Actual:")
print(y[:10].flatten())

print("\nPredicted:")
print(y_pred[:10].flatten())

rmse = np.sqrt(np.mean((y_pred - y) ** 2))

print("RMSE:", rmse)

Actual:
[ 6.28517133 10.10128223  4.04587621 11.48326832  4.40116306  2.49072241
  2.00233338 -1.19657137  2.69301084  0.22186667]

Predicted:
[ 6.30276288  9.96453579  3.84500446 11.34903048  4.68535194  2.68257722
  1.92540094 -1.0824962   2.59446967 -0.38641148]
RMSE: 0.2007167195911209


### Classification Using MLP

In [16]:
import numpy as np

np.random.seed(42)

n = 100

# Class 0
X0 = np.random.randn(n, 2) + np.array([-2, -2])
y0 = np.zeros(n, dtype=int)

# Class 1
X1 = np.random.randn(n, 2) + np.array([2, -2])
y1 = np.ones(n, dtype=int)

# Class 2
X2 = np.random.randn(n, 2) + np.array([0, 2])
y2 = np.full(n, 2, dtype=int)

# Combine
X = np.vstack([X0, X1, X2])
y = np.concatenate([y0, y1, y2])

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFirst 10 X:")
print(X[:10])

print("\nFirst 10 y:")
print(y[:10])

X shape: (300, 2)
y shape: (300,)

First 10 X:
[[-1.50328585 -2.1382643 ]
 [-1.35231146 -0.47697014]
 [-2.23415337 -2.23413696]
 [-0.42078718 -1.23256527]
 [-2.46947439 -1.45743996]
 [-2.46341769 -2.46572975]
 [-1.75803773 -3.91328024]
 [-3.72491783 -2.56228753]
 [-3.01283112 -1.68575267]
 [-2.90802408 -3.4123037 ]]

First 10 y:
[0 0 0 0 0 0 0 0 0 0]


In [17]:
model = MLP(
    architecture=[2, 8, 8, 3],
    X=X,
    y=y,
    epochs=5000,
    lr=0.01,
    use_case="Classification"
)
model.fit()

probabilities = model.forward()

predictions = np.argmax(probabilities, axis=1)

accuracy = np.mean(predictions == y)

print("Accuracy:", accuracy)

Accuracy: 0.9733333333333334
